# Referencia histórica y regresión lineal — validación de marzo de 2011
Lourdes Lema · Proyecto de titulación UDLA · Bloque 2 · v01

**Alcance:** entrenar exclusivamente con junio, septiembre y diciembre de 2010; evaluar marzo de 2011. No se selecciona todavía un modelo definitivo ni se evalúan junio o septiembre de 2011.

**Cómo usar:** subir este notebook a Colab, guardar una copia en Drive y ejecutar las celdas de código en orden. La primera solicita `Evidencias_cortes_v06(1).zip` (también admite el nombre sin `(1)`). No descomprimir. Al finalizar se descarga un ZIP de evidencias: compartirlo para comparar la reproducción.

La verificación local previa no constituye una ejecución realizada por la autora en Colab. Esta copia pública no conserva salidas; al ejecutar el cuaderno se actualizarán los resultados a partir de los archivos de entrada. No es necesario subir la base transaccional.

## Decisiones del bloque
Se conserva el protocolo entregado: cinco predictores; H fuera de X; objetivo neto firmado a 90 días; regresión lineal con intercepto por mínimos cuadrados; sin imputación, escalado, transformaciones, eliminación de extremos, redondeo de cálculo ni truncamiento de predicciones. Cada observación cliente-corte tiene el mismo peso durante el ajuste. Los clientes repetidos se conservan. No se calculan pruebas de significación que presupongan independencia.

El criterio de ajuste minimiza errores cuadrados; el MAE es el criterio principal de evaluación y selección previsto. Son funciones diferentes. Los coeficientes describen asociaciones del modelo, no efectos causales.


## Carga y procedencia
**Propósito:** utilizar los archivos v06 auditados y conservar su identidad.

**Operaciones:** importar bibliotecas, solicitar un solo ZIP y comprobar las huellas de los CSV de entrenamiento y validación. Leer únicamente entrenamiento y las filas de marzo del CSV de validación. El archivo de prueba no se abre.

**Resultado esperado:** 8.939 filas para entrenamiento y 3.338 para marzo.

**Interpretación:** una huella distinta detiene el proceso para revisar la versión. Aunque el CSV de validación contiene junio, solo se convierten en tabla las filas de marzo; junio no interviene en cálculos ni métricas.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, io, json, hashlib, platform, zipfile
import pandas as pd
import numpy as np
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RUTA_ZIP = None  # En Colab dejar None para seleccionar el archivo.
if RUTA_ZIP is None:
    from google.colab import files
    cargados = files.upload()
    candidatos = [n for n in cargados if n.lower().endswith('.zip')]
    if len(candidatos) != 1:
        raise ValueError('Cargue únicamente el ZIP de evidencias de cortes v06.')
    ruta = Path(candidatos[0])
else:
    ruta = Path(RUTA_ZIP)
HUELLAS_CSV = {'entrenamiento_inicial_v06.csv': '65d7dc09069779841830e0d596ecd4eb6cd78a61f193b12d8a7ffc08a356cd59', 'validacion_v06.csv': 'bfb661e6f35357af531beebfba7f6bf0acffeb126a0a259c129ad4479a8bcedc'}
contenido = {}
with zipfile.ZipFile(ruta) as z:
    for nombre, huella in HUELLAS_CSV.items():
        coincidencias = [n for n in z.namelist() if Path(n).name == nombre]
        assert len(coincidencias) == 1, f'Archivo ausente o ambiguo: {nombre}'
        contenido[nombre] = z.read(coincidencias[0])
        assert hashlib.sha256(contenido[nombre]).hexdigest() == huella, f'Versión distinta: {nombre}'
train = pd.read_csv(io.BytesIO(contenido['entrenamiento_inicial_v06.csv']), dtype={'Customer ID':'string'})
lector = csv.DictReader(io.StringIO(contenido['validacion_v06.csv'].decode('utf-8-sig')))
marzo = pd.DataFrame([fila for fila in lector if fila['fecha_corte'][:10] == '2011-03-01'])
print('Entrenamiento:', len(train), '| Validación marzo:', len(marzo))
print('Versión scikit-learn:', sklearn.__version__)


## Tipos, variables y separación temporal
**Propósito:** impedir que identificadores, H o ventas futuras entren como predictores y comprobar las etiquetas disponibles.

**Operaciones:** restaurar tipos, verificar claves únicas, valores finitos, conteos y P=M/F; ordenar por corte e identificador; construir X con exactamente cinco columnas e Y por separado.

**Resultado esperado:** X de entrenamiento de tamaño 8.939 × 5 y X de marzo de 3.338 × 5. Las etiquetas del corte de diciembre terminan el 1 de marzo, instante excluido.

**Interpretación:** puede haber clientes repetidos entre períodos; no se eliminan porque el uso previsto es pronosticar compradores conocidos con historia anterior. La cobertura temporal se auditó desde la base v04 en el bloque anterior.


In [ ]:
predictores = ['recencia_dias','frecuencia_180d','diversidad_180d',
               'importe_neto_180d','importe_neto_por_compra_180d']
objetivo = 'objetivo_neto_90d'
referencia = 'referencia_neto_90d'
variables = predictores + [referencia, objetivo]
controles = []
def verificar(nombre, condicion):
    controles.append({'control':nombre,'aprobado':bool(condicion)})
    if not condicion:
        raise AssertionError(nombre)
for nombre, d, n in [('entrenamiento',train,8939),('marzo',marzo,3338)]:
    verificar(nombre+' columnas',set(d.columns)==set(['Customer ID','fecha_corte']+variables))
    d['Customer ID'] = d['Customer ID'].astype('string')
    d['fecha_corte'] = pd.to_datetime(d['fecha_corte'], errors='raise')
    for c in variables: d[c] = pd.to_numeric(d[c], errors='raise')
    verificar(nombre+' filas',len(d)==n)
    verificar(nombre+' sin nulos',not d.isna().any().any())
    verificar(nombre+' clave única',not d.duplicated(['Customer ID','fecha_corte']).any())
    verificar(nombre+' valores finitos',np.isfinite(d[variables]).all().all())
    verificar(nombre+' compras positivas',d.frecuencia_180d.ge(1).all())
    verificar(nombre+' P=M/F',np.allclose(d.importe_neto_por_compra_180d,
              d.importe_neto_180d/d.frecuencia_180d,rtol=1e-12,atol=1e-9))
    d.sort_values(['fecha_corte','Customer ID'],inplace=True)
    d.reset_index(drop=True,inplace=True)
cortes = pd.to_datetime(['2010-06-01','2010-09-01','2010-12-01'])
verificar('Conteos de entrenamiento',train.groupby('fecha_corte').size().to_dict()==dict(zip(cortes,[2652,2822,3465])))
verificar('Clientes distintos entrenamiento',train['Customer ID'].nunique()==4225)
verificar('Solo marzo en validación',marzo.fecha_corte.eq(pd.Timestamp('2011-03-01')).all())
verificar('Etiquetas disponibles',((train.fecha_corte+pd.Timedelta(days=90))<=pd.Timestamp('2011-03-01')).all())
X_train, y_train = train[predictores].copy(), train[objetivo].copy()
X_marzo, y_marzo = marzo[predictores].copy(), marzo[objetivo].copy()
print('X entrenamiento:', X_train.shape, '| X marzo:', X_marzo.shape)
print('Controles aprobados:',len(controles))


## Referencia H y ajuste de regresión lineal
**Propósito:** comparar una regla sencilla con un modelo que aprende del historial.

**Operaciones:** H predice que las ventas netas de los próximos 90 días serán iguales a las de los 90 anteriores. No necesita entrenamiento. `fit` estima el intercepto y cinco coeficientes usando solo entrenamiento; `predict` aplica esa ecuación a marzo. Se verifica la predicción reconstruyéndola con los coeficientes.

**Resultado esperado:** una predicción de cada alternativa por cliente, y seis parámetros estimados del modelo lineal.

**Interpretación:** las predicciones son importes en GBP. Se permiten valores negativos. El tamaño de los coeficientes no es una clasificación de importancia: las unidades difieren y P depende de M y F. Un rango completo descarta dependencia lineal exacta en esta matriz, pero no garantiza ausencia de colinealidad o estabilidad.


In [ ]:
pred_h = marzo[referencia].to_numpy(copy=True)
modelo = LinearRegression(fit_intercept=True, copy_X=True, positive=False)
modelo.fit(X_train, y_train)
pred_rl = modelo.predict(X_marzo)
verificar('Predicciones finitas',np.isfinite(pred_rl).all() and np.isfinite(pred_h).all())
verificar('Predicción coincide con ecuación',np.allclose(pred_rl,
          X_marzo.to_numpy()@modelo.coef_+modelo.intercept_,rtol=1e-12,atol=1e-8))
coeficientes = pd.DataFrame({'termino':['intercepto']+predictores,
                           'coeficiente':[float(modelo.intercept_)]+modelo.coef_.tolist()})
print(coeficientes.to_string(index=False))
print('Rango de la matriz centrada:',modelo.rank_,'de',len(predictores))
print('Predicciones negativas RL conservadas:',int((pred_rl<0).sum()))


## Evaluación de marzo
**Propósito:** medir el error fuera del entrenamiento en el primer corte de validación.

**Operaciones:** calcular las tres métricas para ambas alternativas sobre los mismos 3.338 clientes; verificar MAE y RMSE mediante sus fórmulas; guardar errores por cliente.

**Resultado esperado:** una tabla con dos filas. Solo se redondea la presentación.

**Interpretación:** MAE es el error absoluto promedio en GBP por cliente a 90 días; menor es mejor. RMSE también se expresa en GBP y da mayor peso a errores grandes. R² compara los errores cuadrados con predecir la media observada de marzo; puede ser negativo y no es un porcentaje de exactitud. Esa media no es un pronóstico disponible al corte. La diferencia MAE(RL)−MAE(H) negativa favorece RL. Una mejora en marzo no demuestra superioridad definitiva ni mejor priorización; faltan junio, Random Forest y la prueba final.


In [ ]:
filas=[]
for nombre,pred in [('Referencia H',pred_h),('Regresión lineal',pred_rl)]:
    mae=float(mean_absolute_error(y_marzo,pred))
    rmse=float(np.sqrt(mean_squared_error(y_marzo,pred)))
    r2=float(r2_score(y_marzo,pred))
    error=y_marzo.to_numpy()-pred
    verificar(nombre+' MAE comprobado',np.isclose(mae,np.mean(np.abs(error)),rtol=1e-12))
    verificar(nombre+' RMSE comprobado',np.isclose(rmse,np.sqrt(np.mean(error**2)),rtol=1e-12))
    filas.append({'modelo':nombre,'corte':'2011-03-01','n':len(marzo),'MAE_GBP':mae,'RMSE_GBP':rmse,'R2':r2})
metricas=pd.DataFrame(filas)
print(metricas.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
delta=float(metricas.iloc[1].MAE_GBP-metricas.iloc[0].MAE_GBP)
print(f'Diferencia MAE RL menos H: {delta:.6f} GBP')
if metricas.iloc[0].MAE_GBP>0:
    print(f'Reducción relativa del MAE frente a H: {-100*delta/metricas.iloc[0].MAE_GBP:.4f}%')
predicciones=marzo[['Customer ID','fecha_corte',objetivo,referencia]].copy()
predicciones['prediccion_RL']=pred_rl
predicciones['error_RL_observado_menos_predicho']=y_marzo.to_numpy()-pred_rl
predicciones['error_H_observado_menos_predicho']=y_marzo.to_numpy()-pred_h
print('Evaluación de marzo completada. No se selecciona todavía modelo definitivo.')


## Guardado de evidencias y bitácora
**Propósito:** permitir reproducir, cotejar y posteriormente integrar este bloque al repositorio.

**Operaciones:** exportar métricas, predicciones, coeficientes, controles, versiones y parámetros; comprimir únicamente los archivos de esta ejecución. El paquete contiene los datos usados para que la comparación sea auditable. El notebook se conserva por separado en Drive.

**Resultado esperado:** descarga de `Evidencias_RL_H_marzo_v01.zip`.

**Interpretación:** compartir ese ZIP para comparar tus resultados con esta ejecución verificada. Diferencias pequeñas pueden proceder de versiones numéricas; cualquier discrepancia se revisará antes de continuar. No contiene resultados de junio, prueba final ni ranking comercial.


In [ ]:
salida=Path('salidas_RL_H_marzo_v01');salida.mkdir(exist_ok=True)
tablas={'metricas_marzo.csv':metricas,'predicciones_marzo.csv':predicciones,
        'coeficientes_RL.csv':coeficientes,'controles.csv':pd.DataFrame(controles),
        'entrenamiento_utilizado.csv':train,'validacion_marzo_utilizada.csv':marzo}
archivos=[]
for nombre,tabla in tablas.items():
    p=salida/nombre;tabla.to_csv(p,index=False);archivos.append(p)
bitacora={'version':'bloque2_v01','fecha_utc':datetime.now(timezone.utc).isoformat(),
 'sha256_zip_entrada':hashlib.sha256(ruta.read_bytes()).hexdigest(),
 'sha256_csv_entrada':HUELLAS_CSV,'predictores':predictores,'objetivo':objetivo,'referencia':referencia,
 'parametros_RL':modelo.get_params(),'rango_X_centrada':int(modelo.rank_),
 'filas_entrenamiento':len(train),'filas_validacion':len(marzo),
 'versiones':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,'scikit-learn':sklearn.__version__},
 'reglas':'Mínimos cuadrados con intercepto, pesos iguales por fila; sin escalado, imputación, filtros o truncamiento.',
 'alcance':'Solo marzo de 2011; no selección definitiva, Random Forest, junio ni prueba final.',
 'procedencia':'Cortes v06 cotejados contra reconstrucción v04 en sesión previa; 17703 claves y siete variables con diferencia cero.',
 'pendientes':['Reproducción Colab por Lourdes y cotejo','Validación junio','Random Forest y selección','Prueba final y priorización'],
 'sha256_salidas':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in archivos}}
p=salida/'bitacora.json';p.write_text(json.dumps(bitacora,ensure_ascii=False,indent=2));archivos.append(p)
paquete=Path('Evidencias_RL_H_marzo_v01.zip')
with zipfile.ZipFile(paquete,'w',zipfile.ZIP_DEFLATED) as z:
    for p in archivos:z.write(p,p.name)
print('Evidencias generadas:',paquete.name)
try:
    from google.colab import files
except ImportError:
    print('Ejecución local verificada; descarga automática solo en Colab.')
else:
    files.download(str(paquete))


## Lectura y defensa del bloque
Antes de continuar, explica con tus palabras:
- ¿Por qué H tiene 90 días mientras los predictores resumen 180 días?
- ¿Por qué el error de marzo es una validación y no el resultado final?
- ¿Qué significa un MAE de determinada cantidad de GBP y por qué no expresa ventas perdidas?
- ¿Por qué se mantienen predicciones negativas y por qué menor MAE no garantiza mejor selección de clientes?

## Fuentes técnicas consultadas
scikit-learn developers. (s. f.). *LinearRegression*. scikit-learn. https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

scikit-learn developers. (s. f.). *Metrics and scoring: Quantifying the quality of predictions*. scikit-learn. https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics

Estas referencias documentan la implementación. Al integrarlas a la tesis, consolidarlas con la bibliografía existente y asignar letras a/b/c si corresponde para entradas del mismo autor y sin fecha. No reemplazan los antecedentes académicos del proyecto.
